# Week 06 Homework - Recipe Assistant Evaluation

Complete implementation of Q1-Q6 and Bonus directly in notebook.
Covers: scenario design, batch evaluation, manual labeling, LLM judge, alignment metrics, judge improvement, and synthetic dataset.

In [2]:
import os
import sys
import csv
import json
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel
from pydantic_ai import Agent
from dotenv import load_dotenv

# Fix nested event loop issue in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except: pass

# Setup paths and env
ROOT = Path('..').resolve()
load_dotenv(ROOT / '.env', override=True)
load_dotenv(ROOT.parent / '.env', override=True)

# Import recipe agent
sys.path.insert(0, str(ROOT))
from recipe_agent import agent

print(f'Root: {ROOT}')
print(f'OpenAI key set: {bool(os.getenv("OPENAI_API_KEY"))}')

Root: D:\AI-Engineering\AI-Engineering-Buildcamp\week 06\homework
OpenAI key set: True


In [3]:
# Utilities
def calculate_cost(usage, model='gpt-4o-mini') -> float:
    prices = {'gpt-4o-mini': {'input': 0.15, 'output': 0.60}}
    p = prices[model]
    return round((usage.input_tokens / 1e6 * p['input']) + (usage.output_tokens / 1e6 * p['output']), 6)

def run_batch(scenarios_df):
    results = []
    for i, (_, row) in enumerate(scenarios_df.iterrows()):
        t0 = time.time()
        res = agent.run_sync(row['question'])
        results.append({
            'question': row['question'],
            'category': row.get('category', ''),
            'type': row.get('type', ''),
            'output': res.output,
            'tokens': {k: getattr(res.usage(), k) for k in ['input_tokens', 'output_tokens', 'total_tokens']},
            'cost': calculate_cost(res.usage()),
            'time': round(time.time() - t0, 2)
        })
        print(f'[{i+1}/{len(scenarios_df)}]', end=' ')
    return results

def judge_batch(results_list, judge_agent):
    judged = []
    for i, r in enumerate(results_list):
        eval_res = judge_agent.run_sync(f"Q: {r['question']}\nA: {r['output']}").output
        judged.append({**r, 'judge_label': eval_res.label, 'judge_reasoning': eval_res.reasoning})
        print(f'[{i+1}/{len(results_list)}]', end=' ')
    return judged

print('✓ Utilities ready')

✓ Utilities ready


In [4]:
# =============================================================================
# Load Scenarios
# =============================================================================

scenarios_path = ROOT / 'scenarios.csv'
scenarios = pd.read_csv(scenarios_path)

print(f'Total scenarios: {len(scenarios)}')
print(f'Categories: {scenarios["category"].unique().tolist()}')
print(f'Types: {scenarios["type"].unique().tolist()}')

# Verify requirements
assert len(scenarios) >= 20, 'Need at least 20 scenarios'
has_almond = scenarios['question'].str.contains('almond milk', case=False, na=False).any()
assert has_almond, 'Add almond milk scenario'

print('\n✓ Scenarios validated')

Total scenarios: 21
Categories: ['specific-recipe', 'ingredient-search', 'preference', 'missing-recipe', 'recipe-detail', 'cuisine-search', 'substitution', 'dietary', 'nutrition', 'out-of-scope', 'edge-case']
Types: ['direct', 'ingredient-based', 'vague', 'not-in-collection', 'specific', 'short-query', 'unsupported-detail', 'ingredient-check', 'not-in-data', 'non-recipe', 'non-cooking', 'recipe-constraint', 'preference', 'ingredient-list']

✓ Scenarios validated


## Q1: Scenario Design - Test Almond Milk Hallucination

In [5]:
# Q1: Almond Milk Test
resp = agent.run_sync('Can I substitute almond milk in the pancakes?')
text_lower = resp.output.lower()
advice = any(w in text_lower for w in ['you can', 'substitute', 'ratio', 'texture'])
refuses = any(w in text_lower for w in ['do not have', "don't have", 'not in'])
hallucinated = advice and not refuses
print(f'Q1: Hallucinated = {hallucinated}\n{resp.output}')

RuntimeError: This event loop is already running

In [ ]:
# Q2: Batch Run
results = run_batch(scenarios)
(ROOT / 'results.json').write_text(json.dumps(results, indent=2))
df_results = pd.DataFrame(results)
total_cost = sum(r['cost'] for r in results)
almond = df_results[df_results['question'].str.contains('almond', case=False)].iloc[0]
print(f'\nTotal cost: ${total_cost:.6f}')
print(f'Q2: Almond milk cost = ${almond["cost"]:.6f}')

## Judge Configuration & Instructions

## Judge Instructions

**Good**: Accurate answers using ONLY recipe collection info. Properly declines out-of-scope questions.

**Bad**: Hallucination (makes up recipes, substitutions, or cooking tips not in data). Wrong facts. Claims no recipe exists when it does.

In [ ]:
# Q3: Labeling Template
labels_path = ROOT / 'human_labels.csv'
if not labels_path.exists():
    df_results[['question']].assign(label='', reason='').to_csv(labels_path, index=False)
    print(f'Q3: Template created at {labels_path}')
    print('Fill in "label" (good/bad) and "reason" columns')
else:
    print(f'Q3: {labels_path} exists')

In [ ]:
# Q4: Judge V1
from pydantic import BaseModel
from typing import Literal

class JudgeEvaluation(BaseModel):
    label: Literal['good', 'bad']
    reasoning: str

judge_instr = "Evaluate if response uses ONLY recipe data. Bad = hallucination, wrong facts, or out-of-scope answers."
judge_v1 = Agent('openai:gpt-4o-mini', output_type=JudgeEvaluation, instructions=judge_instr)

print('Running judge V1...')
judged = judge_batch(results, judge_v1)
(ROOT / 'results_judged.json').write_text(json.dumps(judged, indent=2))
bad_count = sum(1 for r in judged if r['judge_label'] == 'bad')
print(f'\nQ4: Judge bad count = {bad_count}')

## Q5: Alignment Metrics & Q6: Judge Improvement

In [ ]:
# Q5: Alignment (needs human labels)
if labels_path.exists():
    human = pd.read_csv(labels_path)
    merged = pd.DataFrame(judged).merge(human[['question', 'label']], on='question')
    merged['h_bad'] = merged['label'] == 'bad'
    merged['j_bad'] = merged['judge_label'] == 'bad'
    tp = (merged['h_bad'] & merged['j_bad']).sum()
    fp = (~merged['h_bad'] & merged['j_bad']).sum()
    fn = (merged['h_bad'] & ~merged['j_bad']).sum()
    acc = (tp + (merged['h_bad'] == merged['j_bad']).sum()) / len(merged) if len(merged) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f'Q5: Accuracy={acc:.2f}, Precision={prec:.2f}, Recall={rec:.2f}')
else:
    print('Q5: Fill human_labels.csv first')

## Q6: Judge Improvement

In [ ]:
# Q6: Judge V2 (Improved)
judge_instr_v2 = "STRICT: Flag hallucinations (made-up advice, cooking tips, numbers not in recipes). Bad = any made-up info or wrong facts."
judge_v2 = Agent('openai:gpt-4o-mini', output_type=JudgeEvaluation, instructions=judge_instr_v2)

print('Running judge V2...')
judged_v2 = judge_batch(results, judge_v2)
bad_v2 = sum(1 for r in judged_v2 if r['judge_label'] == 'bad')
bad_v1 = sum(1 for r in judged if r['judge_label'] == 'bad')
print(f'\nQ6: V1 bad={bad_v1}, V2 bad={bad_v2}, Diff={abs(bad_v1-bad_v2)}')

## Bonus: Synthetic Dataset Evaluation

In [ ]:
# Bonus: Generate Synthetic Questions
recipes = json.loads((ROOT / 'recipes.json').read_text())

class SyntheticQuestions(BaseModel):
    questions: list[dict]

gen_instr = "Generate 5 diverse questions about this recipe: 1 factual, 1 how-to, 1 detail, 1 substitution, 1 comparison."
gen = Agent('openai:gpt-4o-mini', output_type=SyntheticQuestions, instructions=gen_instr)

synthetic = []
for i, r in enumerate(recipes[:15]):
    prompt = f"Recipe: {r['name']}\nCuisine: {r['cuisine']}\nIngredients: {', '.join(r['ingredients'])}"
    try:
        res = gen.run_sync(prompt).output
        synthetic.extend(res.questions)
        print(f'[{i+1}/15]', end=' ')
    except: pass

print(f'\nGenerated {len(synthetic)} synthetic questions')

# Run through agent and judge
syn_results = []
for q in synthetic:
    res = agent.run_sync(q.get('question', ''))
    syn_results.append({'question': q.get('question', ''), 'output': res.output, 'cost': calculate_cost(res.usage())})

print('Evaluating synthetic...')
syn_judged = judge_batch(syn_results, judge_v1)
bad_pct = sum(1 for r in syn_judged if r['judge_label'] == 'bad') / len(syn_judged) * 100 if syn_judged else 0
print(f'Bonus: {bad_pct:.1f}% synthetic responses marked bad')

## Final Summary

In [ ]:
# Summary
print('='*50)
print('WEEK 06 RESULTS')
print('='*50)
print(f'Q1: Hallucination = {hallucinated}')
print(f'Q2: Almond cost = ${almond["cost"]:.6f}')
print(f'Q3: Template created')
print(f'Q4: Judge bad = {bad_count}')
print(f'Q5: Accuracy = {acc:.2f}' if 'acc' in locals() else 'Q5: Fill labels first')
print(f'Q6: V1={bad_v1}, V2={bad_v2}')
print(f'Bonus: {bad_pct:.1f}% bad')
print('='*50)